In [1]:
import heapq

graph = {
    'S': [('A', 2), ('B', 4), ('C', 3)],
    'A': [('D', 5), ('E', 6)],
    'B': [('F', 2), ('G', 4)],
    'C': [('H', 7), ('I', 3)],
    'D': [('J', 4)],
    'E': [('K', 3)],
    'F': [('L', 6)],
    'G': [('M', 5)],
    'H': [('N', 2)],
    'I': [('O', 4)],
    'J': [], 'K': [], 'L': [], 'M': [],
    'N': [], 'O': []
}

def dynamic_beam_search(start, goal):
    beam_width = 2
    max_beam_width = 5
    level = 0

    beam = [(0, [start])]

    while beam:
        print(f"\nLevel {level}")
        print(f"Current Beam Width: {beam_width}")
        print("Beam Nodes:", [path for cost, path in beam])

        candidates = []

        for cost, path in beam:
            current = path[-1]

            if current == goal:
                print("\nGoal Found")
                return path, cost

            for neighbor, edge_cost in graph.get(current, []):
                new_cost = cost + edge_cost
                new_path = path + [neighbor]
                candidates.append((new_cost, new_path))

        if not candidates:
            break

        beam = heapq.nsmallest(beam_width, candidates, key=lambda x: x[0])

        level += 1

        if level % 3 == 0 and beam_width < max_beam_width:
            beam_width += 1
            print(f"Beam width increased to {beam_width}")

    return None, float('inf')


start_node = 'S'
goal_node = 'O'

path, cost = dynamic_beam_search(start_node, goal_node)

print("\nFinal Result:")
if path:
    print("Path found:", " → ".join(path))
    print("Total cost:", cost)
else:
    print("No path found")



Level 0
Current Beam Width: 2
Beam Nodes: [['S']]

Level 1
Current Beam Width: 2
Beam Nodes: [['S', 'A'], ['S', 'C']]

Level 2
Current Beam Width: 2
Beam Nodes: [['S', 'C', 'I'], ['S', 'A', 'D']]
Beam width increased to 3

Level 3
Current Beam Width: 3
Beam Nodes: [['S', 'C', 'I', 'O'], ['S', 'A', 'D', 'J']]

Goal Found

Final Result:
Path found: S → C → I → O
Total cost: 10


In [2]:
import random

def calculate_conflicts(state):
    conflicts = 0
    n = len(state)

    for i in range(n):
        for j in range(i + 1, n):
            if state[i] == state[j] or abs(state[i] - state[j]) == abs(i - j):
                conflicts += 1

    return conflicts


def get_neighbors(state):
    neighbors = []
    n = len(state)

    for row in range(n):
        for col in range(n):
            if col != state[row]:
                new_state = list(state)
                new_state[row] = col
                neighbors.append(new_state)

    return neighbors


def hill_climbing(initial_state):
    current_state = initial_state
    current_conflicts = calculate_conflicts(current_state)

    while True:
        neighbors = get_neighbors(current_state)

        best_neighbor = current_state
        best_conflicts = current_conflicts

        for neighbor in neighbors:
            conflicts = calculate_conflicts(neighbor)
            if conflicts < best_conflicts:
                best_neighbor = neighbor
                best_conflicts = conflicts

        if best_conflicts >= current_conflicts:
            return current_state, current_conflicts

        current_state = best_neighbor
        current_conflicts = best_conflicts


def random_restart_hill_climbing(n, max_restarts=20):
    for attempt in range(1, max_restarts + 1):
        initial_state = [random.randint(0, n - 1) for _ in range(n)]

        print(f"\nRestart {attempt}: Initial State = {initial_state}")

        solution, conflicts = hill_climbing(initial_state)

        print(f"Final State = {solution}")
        print(f"Conflicts = {conflicts}")

        if conflicts == 0:
            print("\nSolution Found")
            return solution, conflicts, attempt

    print("\nFailed to find solution within restart limit")
    return None, None, max_restarts


n = 8
solution, conflicts, attempts = random_restart_hill_climbing(n)

print("\nFinal Result:")
if solution:
    print(f"Solution: {solution}")
    print(f"Found in {attempts} restart(s)")
else:
    print("No solution found")



Restart 1: Initial State = [1, 1, 2, 2, 0, 6, 1, 2]
Final State = [3, 7, 4, 2, 0, 6, 1, 5]
Conflicts = 0

Solution Found

Final Result:
Solution: [3, 7, 4, 2, 0, 6, 1, 5]
Found in 1 restart(s)


In [3]:
import random

teachers = ['T1', 'T2', 'T3', 'T4', 'T5']
courses = ['C1', 'C2', 'C3', 'C4', 'C5']

slots_per_day = 5
days = 5
total_slots = slots_per_day * days 

population_size = 20
mutation_rate = 0.1
generations = 200


def create_chromosome():
    chromosome = []

    course_list = []
    for c in courses:
        course_list += [c] * 3

    while len(course_list) < total_slots:
        course_list.append(random.choice(courses))

    random.shuffle(course_list)

    for c in course_list:
        t = random.choice(teachers)
        chromosome.append((c, t))

    return chromosome


def fitness(chromosome):
    penalty = 0

    for day in range(days):
        start = day * slots_per_day
        end = start + slots_per_day
        day_slots = chromosome[start:end]

        teachers_in_slot = [t for (c, t) in day_slots]
        if len(teachers_in_slot) != len(set(teachers_in_slot)):
            penalty += 10  

    course_count = {}
    for c, t in chromosome:
        course_count[c] = course_count.get(c, 0) + 1

    for c in courses:
        penalty += abs(course_count.get(c, 0) - 3) * 5

    for t in teachers:
        count = 0
        for slot in chromosome:
            if slot[1] == t:
                count += 1
                if count > 3:
                    penalty += 5
            else:
                count = 0

    return penalty


def selection(population):
    population.sort(key=lambda x: fitness(x))
    return population[:len(population)//2]


def crossover(p1, p2):
    point = random.randint(1, total_slots - 2)
    return p1[:point] + p2[point:]


def mutate(chromosome):
    i, j = random.sample(range(total_slots), 2)
    chromosome[i], chromosome[j] = chromosome[j], chromosome[i]
    return chromosome


def genetic_algorithm():
    population = [create_chromosome() for _ in range(population_size)]

    for gen in range(generations):
        fitness_scores = [fitness(ch) for ch in population]
        best = min(fitness_scores)

        print(f"Generation {gen+1}, Best Fitness: {best}")

        if best == 0:
            break

        parents = selection(population)

        new_population = []

        while len(new_population) < population_size:
            p1, p2 = random.sample(parents, 2)
            child = crossover(p1, p2)

            if random.random() < mutation_rate:
                child = mutate(child)

            new_population.append(child)

        population = new_population

    best_chromosome = min(population, key=fitness)
    return best_chromosome, fitness(best_chromosome)


best_solution, best_score = genetic_algorithm()

print("\nFinal Best Timetable:")
for i, slot in enumerate(best_solution):
    day = i // slots_per_day + 1
    time = i % slots_per_day + 1
    print(f"Day {day}, Slot {time}: {slot}")

print("\nFinal Fitness (Penalty):", best_score)


Generation 1, Best Fitness: 90
Generation 2, Best Fitness: 90
Generation 3, Best Fitness: 80
Generation 4, Best Fitness: 80
Generation 5, Best Fitness: 80
Generation 6, Best Fitness: 80
Generation 7, Best Fitness: 70
Generation 8, Best Fitness: 70
Generation 9, Best Fitness: 70
Generation 10, Best Fitness: 70
Generation 11, Best Fitness: 60
Generation 12, Best Fitness: 60
Generation 13, Best Fitness: 60
Generation 14, Best Fitness: 60
Generation 15, Best Fitness: 60
Generation 16, Best Fitness: 60
Generation 17, Best Fitness: 60
Generation 18, Best Fitness: 60
Generation 19, Best Fitness: 60
Generation 20, Best Fitness: 50
Generation 21, Best Fitness: 50
Generation 22, Best Fitness: 50
Generation 23, Best Fitness: 50
Generation 24, Best Fitness: 50
Generation 25, Best Fitness: 50
Generation 26, Best Fitness: 50
Generation 27, Best Fitness: 50
Generation 28, Best Fitness: 50
Generation 29, Best Fitness: 50
Generation 30, Best Fitness: 50
Generation 31, Best Fitness: 50
Generation 32, Be